# Khayal: deep learning vs the rule for decline detection

This notebook reproduces `ml/results/dl_decline.md`: GRU, LSTM and an MLP baseline trained on simulated
patients, compared with the app's rule (variant B) on the same test patients.

It runs `python -m ml.dl.run` from the repository, so the notebook and the command line always give the same
numbers. Everything runs on CPU; a GPU is not needed.

- **Full run** (600 patients per profile, 90 days, 5 seeds): about 2 hours on a laptop CPU, similar on Colab.
- **Quick run** (`QUICK = True` below): a few minutes, for checking that everything works. Its numbers are not
  the reported ones.

## 1. Get the code

If the GitHub repository is private, add a Colab secret named `GITHUB_TOKEN` (key icon on the left) holding a
fine-grained token with read access to the repository. A public repository needs nothing.

In [ ]:
import os, subprocess

REPO = "github.com/jansiddiqui/Khayal.git"
BRANCH = "dl-decline"

token = None
try:
    from google.colab import userdata  # only on Colab
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    pass

if not os.path.exists("Khayal"):
    url = f"https://{token}@{REPO}" if token else f"https://{REPO}"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, "Khayal"], check=True)
%cd Khayal

## 2. Check the environment

Colab already has PyTorch, NumPy and Matplotlib, so nothing is installed here. Locally, use
`pip install -r ml/requirements-dl.txt` instead.

In [ ]:
import torch, numpy, matplotlib
print("torch", torch.__version__, "| numpy", numpy.__version__, "| matplotlib", matplotlib.__version__)

## 3. Run the experiment

Each seed simulates a new cohort, splits it 70/15/15 by patient, fits the scalers on the training patients
only, trains the models and scores every method (including the rule) on the same test patients, then tests
the trained models on five shifted simulators without retraining.

Trained models and each finished seed are saved under `data/dl/`. If Colab disconnects, run the cells again:
finished seeds are loaded instead of retrained.

In [ ]:
QUICK = False  # True: a small run to check everything works (numbers are not the reported ones)

if QUICK:
    args = "--per-profile 60 --seeds 2 --epochs 5 --out ml/results/quick --cache data/dl-quick"
    out_dir = "ml/results/quick"
else:
    args = "--per-profile 600 --days 90 --seeds 5 --out ml/results --cache data/dl"
    out_dir = "ml/results"

!python -u -m ml.dl.run {args}

## 4. Results

In [ ]:
from IPython.display import Image, Markdown, display

display(Markdown(open(f"{out_dir}/dl_decline.md", encoding="utf-8").read()))

In [ ]:
for name in ("dl_tradeoff.png", "dl_confusion.png", "dl_delay.png", "dl_training_curves.png"):
    display(Image(filename=f"{out_dir}/{name}"))

## Notes

- The data is simulated (`ml/decline/simulator.py`); study time is an assumption, so the GRU is also reported
  without it.
- The models' early-warning delays depend on the training target (see the Method section of the results).
  The "How early each method tells the profile" table compares methods without that target.
- Per-seed values are in `dl_decline.csv` (long format: method, seed, metric, group, value).